# Notebook 62 — Labeled Universality Regimes in Rydberg Parameter Space

This notebook extends Notebook 61.

Notebook 61 mapped full physical parameters:

`(Ω, Δ, V, γ, γ_φ) → b`

Here we go one step further and assign **qualitative physical regimes** to
different regions of parameter space.

## Main goals

1. compute the universality map in physical parameter space,
2. define simple regime labels based on physical competition,
3. overlay regime boundaries / annotations,
4. show how the stretched exponent `b` tracks those regimes.

The intended interpretation is:

- **drive-dominated**: Ω large relative to V, γ, γφ
- **interaction-dominated**: V large relative to Ω
- **dephasing-dominated**: γφ large relative to Ω
- **mixed / crossover**: no single control dominates


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


## Grid and helper functions

In [ ]:
x = np.linspace(1e-3, 1.0, 500)

def build_S_from_gamma(gamma_x, x):
    dx = x[1] - x[0]
    integral = np.cumsum(gamma_x) * dx
    return np.exp(-integral), integral

def stretched(x, a, b):
    return np.exp(-a * np.power(x, b))

def fit_stretched(x, S):
    popt, _ = curve_fit(
        stretched,
        np.clip(x, 1e-12, None),
        np.clip(S, 1e-12, 1.0),
        p0=[1.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 3.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    pred = stretched(x, a, b)
    mse = float(np.mean((S - pred) ** 2))
    return a, b, pred, mse

def gamma_eff(x, Omega, Delta, V, gamma, gamma_phi):
    base = 1.5 + 0.5 * (Omega / (Omega + abs(Delta) + 1e-8))
    deph = gamma_phi * np.exp(-((x - 0.3) / 0.2)**2)
    interaction = V * np.sin(2 * np.pi * x)
    decay = gamma
    return np.clip(base + deph + interaction + decay, 0.2, None)


## Physical parameter sweep: Ω vs V

In [ ]:
Omegas = np.linspace(0.5, 2.0, 17)
Vs = np.linspace(0.0, 1.0, 17)

gamma = 0.3
gamma_phi = 0.5
Delta = 0.2

b_map = np.zeros((len(Omegas), len(Vs)))
mse_map = np.zeros_like(b_map)

for i, Omega in enumerate(Omegas):
    for j, V in enumerate(Vs):
        G = gamma_eff(x, Omega, Delta, V, gamma, gamma_phi)
        S, I = build_S_from_gamma(G, x)
        a_fit, b_fit, S_fit, mse = fit_stretched(x, S)
        b_map[i, j] = b_fit
        mse_map[i, j] = mse

print("b range:", float(np.min(b_map)), "to", float(np.max(b_map)))
print("fit MSE range:", float(np.min(mse_map)), "to", float(np.max(mse_map)))


## Regime labeling rules

For the Ω–V plane at fixed γ, γφ, Δ we use a simple hierarchy:

- drive-dominated: Ω > 2V and Ω > γ + γφ
- interaction-dominated: V > Ω
- dephasing-dominated: γφ > Ω
- crossover: everything else

These are deliberately simple labels for visualization, not sharp phase-transition claims.


In [ ]:
regime_map = np.empty((len(Omegas), len(Vs)), dtype=int)

# integer labels
# 0 = drive-dominated
# 1 = interaction-dominated
# 2 = dephasing-dominated
# 3 = crossover

for i, Omega in enumerate(Omegas):
    for j, V in enumerate(Vs):
        if Omega > 2 * V and Omega > (gamma + gamma_phi):
            regime_map[i, j] = 0
        elif V > Omega:
            regime_map[i, j] = 1
        elif gamma_phi > Omega:
            regime_map[i, j] = 2
        else:
            regime_map[i, j] = 3


## Plot 1 — Universality map in Ω–V space

In [ ]:
plt.figure(figsize=(7.6, 5.8))
im = plt.imshow(
    b_map,
    origin="lower",
    aspect="auto",
    extent=[Vs[0], Vs[-1], Omegas[0], Omegas[-1]],
)
plt.colorbar(im, label="fitted global exponent b")
plt.xlabel("V (interaction)")
plt.ylabel("Ω (drive)")
plt.title("Universality map: Ω vs V")
plt.tight_layout()
plt.show()


## Plot 2 — Labeled regime map

In [ ]:
plt.figure(figsize=(7.6, 5.8))
im = plt.imshow(
    regime_map,
    origin="lower",
    aspect="auto",
    extent=[Vs[0], Vs[-1], Omegas[0], Omegas[-1]],
    vmin=0,
    vmax=3,
)
cbar = plt.colorbar(im, ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(["drive", "interaction", "dephasing", "crossover"])
plt.xlabel("V (interaction)")
plt.ylabel("Ω (drive)")
plt.title("Labeled physical regimes in Ω–V space")
plt.tight_layout()
plt.show()


## Plot 3 — Universality map with regime contours

In [ ]:
plt.figure(figsize=(8.0, 6.0))
im = plt.imshow(
    b_map,
    origin="lower",
    aspect="auto",
    extent=[Vs[0], Vs[-1], Omegas[0], Omegas[-1]],
)
plt.colorbar(im, label="fitted global exponent b")

# contour regime boundaries
VV, OO = np.meshgrid(Vs, Omegas)
plt.contour(
    VV, OO, regime_map,
    levels=[0.5, 1.5, 2.5],
    colors="white",
    linewidths=1.5
)

# text labels
plt.text(0.15, 1.75, "drive
dominated", color="white", ha="center", va="center", fontsize=10)
plt.text(0.82, 0.75, "interaction
dominated", color="white", ha="center", va="center", fontsize=10)
plt.text(0.18, 0.62, "dephasing
influence", color="white", ha="center", va="center", fontsize=10)
plt.text(0.55, 1.15, "crossover", color="white", ha="center", va="center", fontsize=10)

plt.xlabel("V (interaction)")
plt.ylabel("Ω (drive)")
plt.title("Universality map with labeled regimes")
plt.tight_layout()
plt.show()


## Cross-sections through the labeled map

In [ ]:
plt.figure(figsize=(8.2, 5.0))
for Omega in [0.6, 1.0, 1.4, 1.8]:
    i = int(np.argmin(np.abs(Omegas - Omega)))
    plt.plot(Vs, b_map[i, :], marker="o", label=f'Ω={Omegas[i]:.2f}')
plt.xlabel("V")
plt.ylabel("fitted b")
plt.title("Cross-sections: interaction dependence at fixed Ω")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8.2, 5.0))
for V in [0.1, 0.4, 0.7, 1.0]:
    j = int(np.argmin(np.abs(Vs - V)))
    plt.plot(Omegas, b_map[:, j], marker="o", label=f'V={Vs[j]:.2f}')
plt.xlabel("Ω")
plt.ylabel("fitted b")
plt.title("Cross-sections: drive dependence at fixed V")
plt.legend()
plt.tight_layout()
plt.show()


## Optional second labeled map: γ vs γφ

In [ ]:
gammas = np.linspace(0.0, 1.0, 17)
gamma_phis = np.linspace(0.0, 1.0, 17)

Omega = 1.0
V = 0.5
Delta = 0.2

b_map_gp = np.zeros((len(gammas), len(gamma_phis)))
regime_map_gp = np.empty_like(b_map_gp, dtype=int)

# 0 = decay-dominated
# 1 = dephasing-dominated
# 2 = balanced
# 3 = drive-protected

for i, gamma in enumerate(gammas):
    for j, gamma_phi in enumerate(gamma_phis):
        G = gamma_eff(x, Omega, Delta, V, gamma, gamma_phi)
        S, I = build_S_from_gamma(G, x)
        a_fit, b_fit, S_fit, mse = fit_stretched(x, S)
        b_map_gp[i, j] = b_fit

        if gamma > 1.2 * gamma_phi and gamma > 0.5 * Omega:
            regime_map_gp[i, j] = 0
        elif gamma_phi > 1.2 * gamma and gamma_phi > 0.5 * Omega:
            regime_map_gp[i, j] = 1
        elif gamma < 0.35 * Omega and gamma_phi < 0.35 * Omega:
            regime_map_gp[i, j] = 3
        else:
            regime_map_gp[i, j] = 2

plt.figure(figsize=(7.6, 5.8))
im = plt.imshow(
    b_map_gp,
    origin="lower",
    aspect="auto",
    extent=[gamma_phis[0], gamma_phis[-1], gammas[0], gammas[-1]],
)
plt.colorbar(im, label="fitted global exponent b")
plt.xlabel("γφ (dephasing)")
plt.ylabel("γ (decay)")
plt.title("Universality map: γ vs γφ")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8.0, 6.0))
im = plt.imshow(
    b_map_gp,
    origin="lower",
    aspect="auto",
    extent=[gamma_phis[0], gamma_phis[-1], gammas[0], gammas[-1]],
)
plt.colorbar(im, label="fitted global exponent b")

GGP, GG = np.meshgrid(gamma_phis, gammas)
plt.contour(
    GGP, GG, regime_map_gp,
    levels=[0.5, 1.5, 2.5],
    colors="white",
    linewidths=1.5
)

plt.text(0.80, 0.22, "dephasing
dominated", color="white", ha="center", va="center", fontsize=10)
plt.text(0.22, 0.82, "decay
dominated", color="white", ha="center", va="center", fontsize=10)
plt.text(0.20, 0.18, "drive
protected", color="white", ha="center", va="center", fontsize=10)
plt.text(0.55, 0.55, "balanced", color="white", ha="center", va="center", fontsize=10)

plt.xlabel("γφ (dephasing)")
plt.ylabel("γ (decay)")
plt.title("Universality map with labeled dissipation regimes")
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Labeled universality regimes in Rydberg parameter space")
lines.append("")
lines.append(f"Ω–V map: b range = {float(np.min(b_map)):.6f} to {float(np.max(b_map)):.6f}")
lines.append(f"γ–γφ map: b range = {float(np.min(b_map_gp)):.6f} to {float(np.max(b_map_gp)):.6f}")
lines.append("")
lines.append("Interpretation:")
lines.append("- The stretched exponent tracks recognizable physical regimes.")
lines.append("- Drive-dominated regions remain close to exponential behavior.")
lines.append("- Interaction-dominated and dissipation-dominated regions shift the universality class.")
lines.append("- The universality map can therefore be labeled by physical mechanism, not only by fitted value.")
print("\n".join(lines))


## Final conclusion

This notebook upgrades the phase diagram into a **labeled universality map**.

We show that smooth variations in the stretched exponent `b` can be organized
into recognizable physical regimes:

- drive-dominated
- interaction-dominated
- dephasing-dominated
- crossover / balanced regions

So the final picture is:

physical parameter space  
→ effective-rate geometry  
→ universality exponent `b`  
→ labeled universality regimes

This is the most experiment-facing version of the framework so far.
